In [ ]:
import os
import shutil
import pandas as pd
import random
random.seed(42)

In [ ]:


# ------------------------
# Paths
# ------------------------
DATASET_ROOT = "/home/c/choton/beemachine/datasets/Beemachine"

DIRS = {
    split: {
        "images": os.path.join(DATASET_ROOT, split, "images"),
        "labels": os.path.join(DATASET_ROOT, split, "labels"),
        "masks":  os.path.join(DATASET_ROOT, split, "masks"),
    }
    for split in ["train", "valid", "test"]
}

CSV_PATHS = {
    "train": os.path.join(DATASET_ROOT, "train_labels.csv"),
    "valid": os.path.join(DATASET_ROOT, "valid_labels.csv"),
    "test":  os.path.join(DATASET_ROOT, "test_labels.csv"),
}

# ------------------------
# Load CSVs
# ------------------------
dfs = {k: pd.read_csv(v) for k, v in CSV_PATHS.items()}

# ------------------------
# Helpers
# ------------------------
def move_file_safe(src, dst):
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(src, dst)

def move_sample(image_name, src_split, dst_split):
    """
    Move image, YOLO label (.txt), and mask (_m.png) together.
    """
    stem, _ = os.path.splitext(image_name)

    # Image
    move_file_safe(
        os.path.join(DIRS[src_split]["images"], image_name),
        os.path.join(DIRS[dst_split]["images"], image_name),
    )

    # Label
    move_file_safe(
        os.path.join(DIRS[src_split]["labels"], f"{stem}.txt"),
        os.path.join(DIRS[dst_split]["labels"], f"{stem}.txt"),
    )

    # Mask
    move_file_safe(
        os.path.join(DIRS[src_split]["masks"], f"{stem}_m.png"),
        os.path.join(DIRS[dst_split]["masks"], f"{stem}_m.png"),
    )

def safe_train_classes(train_df, min_count=2):
    return (
        train_df["class"]
        .value_counts()
        .loc[lambda x: x >= min_count]
        .index
        .tolist()
    )

# ------------------------
# Main fixing function
# ------------------------
def fix_split(split_name):
    train_df = dfs["train"]
    split_df = dfs[split_name]

    train_classes = set(train_df["class"])
    split_classes = set(split_df["class"])

    missing_classes = split_classes - train_classes
    if not missing_classes:
        print(f"✅ {split_name}: no missing classes")
        return

    print(f"⚠️  {split_name}: fixing classes {missing_classes}")

    # ---- Move missing-class samples INTO train ----
    to_train = split_df[split_df["class"].isin(missing_classes)]

    for _, row in to_train.iterrows():
        move_sample(row.image_name, split_name, "train")

    dfs["train"] = pd.concat([dfs["train"], to_train], ignore_index=True)
    dfs[split_name] = split_df.drop(to_train.index)

    n_to_replace = len(to_train)

    # ---- Move same number OUT of train ----
    safe_classes = safe_train_classes(dfs["train"])
    candidates = dfs["train"][dfs["train"]["class"].isin(safe_classes)]

    if len(candidates) < n_to_replace:
        raise RuntimeError("❌ Not enough safe samples to rebalance!")

    to_split = candidates.sample(n=n_to_replace, random_state=42)

    for _, row in to_split.iterrows():
        move_sample(row.image_name, "train", split_name)

    dfs[split_name] = pd.concat([dfs[split_name], to_split], ignore_index=True)
    dfs["train"] = dfs["train"].drop(to_split.index)

    print(f"✅ {split_name}: moved {n_to_replace} samples")

# ------------------------
# Run
# ------------------------
fix_split("valid")
fix_split("test")

# ------------------------
# Save updated CSVs
# ------------------------
for split, df in dfs.items():
    df.to_csv(CSV_PATHS[split], index=False)

print("🎉 Images, labels, and masks successfully rebalanced!")
